# getting ready

In [34]:
from common import *

In [35]:

export_date = datetime.now(timezone.utc).date()
date_string = export_date.strftime("%m_%d_%Y")

movie_export_url = (
    f"https://files.tmdb.org/p/exports/"
    f"movie_ids_{date_string}.json.gz"
)

print(movie_export_url)


https://files.tmdb.org/p/exports/movie_ids_11_14_2026.json.gz


In [36]:
response = requests.get(movie_export_url,stream=True,timeout=(20,180),headers={"Accept-Encoding": "identity"})

response.raise_for_status()

print(f'Status code: {response.status_code}')
print(f'Content type: {response.headers.get('Content-type')}')
print(f'Status code: {response.headers.get('Content-length')}')

Status code: 200
Content type: application/octet-stream
Status code: 28298356


Status code: 200
Content type: application/octet-stream
Status code: 28298356

In [37]:

with gzip.GzipFile(fileobj=response.raw) as gz:
    for i, line in enumerate(gz):
        record = json.loads(line)

        print(record)

        if i == 4:
            break


{'adult': False, 'id': 3924, 'original_title': 'Blondie', 'popularity': 0.936, 'video': False}
{'adult': False, 'id': 8773, 'original_title': "L'Amour à vingt ans", 'popularity': 1.1889, 'video': False}
{'adult': False, 'id': 25449, 'original_title': 'New World Disorder 9: Never Enough', 'popularity': 0.48, 'video': False}
{'adult': False, 'id': 31975, 'original_title': 'Sesame Street: Elmo Loves You!', 'popularity': 0.0757, 'video': True}
{'adult': False, 'id': 2, 'original_title': 'Ariel', 'popularity': 1.8002, 'video': False}


{'adult': False, 'id': 3924, 'original_title': 'Blondie', 'popularity': 0.936, 'video': False}
{'adult': False, 'id': 8773, 'original_title': "L'Amour à vingt ans", 'popularity': 1.1889, 'video': False}
{'adult': False, 'id': 25449, 'original_title': 'New World Disorder 9: Never Enough', 'popularity': 0.48, 'video': False}
{'adult': False, 'id': 31975, 'original_title': 'Sesame Street: Elmo Loves You!', 'popularity': 0.0757, 'video': True}
{'adult': False, 'id': 2, 'original_title': 'Ariel', 'popularity': 1.8002, 'video': False}

In [38]:
print(f'response closed: {response.raw.closed}')

response closed: False


In [39]:

response.close()

print(f"Response closed: {response.raw.closed}")


Response closed: True


# trying movies

In [40]:

response = requests.get(
    movie_export_url,
    stream=True,
    timeout=(20, 180),
    headers={"Accept-Encoding": "identity",
             "User-Agent": "CineMatch/1.0"}
)

response.raise_for_status()

print("Connection:", response.status_code)


Connection: 200


In [41]:
total_record = 0
unique_ids = set()
duplicate_ids = 0
adult_record = 0
missing_ids = 0

In [42]:
with gzip.GzipFile(fileobj=response.raw) as gz:
    for line in gz:
        record = json.loads(line)

        total_record +=1

        movie_id = record.get('id')

        if movie_id is None:
            missing_ids +=1


        elif movie_id in unique_ids:
            duplicate_ids +=1

        else:
            unique_ids.add(movie_id)

        if record.get('adult') is True:
            adult_record +=1

response.close()


In [43]:

print(f"Total records: {total_record:,}")
print(f"Unique IDs: {len(unique_ids):,}")
print(f"Duplicate IDs: {duplicate_ids:,}")
print(f"Adult records: {adult_record:,}")
print(f"Missing IDs: {missing_ids:,}")


Total records: 1,275,504
Unique IDs: 1,275,504
Duplicate IDs: 0
Adult records: 0
Missing IDs: 0


Total records: 1,275,504
Unique IDs: 1,275,504
Duplicate IDs: 0
Adult records: 0
Missing IDs: 0

# trying tv series

In [44]:
tv_export_url = (
    f"https://files.tmdb.org/p/exports/"
    f"tv_series_ids_{date_string}.json.gz"
)

In [45]:
print(tv_export_url)

https://files.tmdb.org/p/exports/tv_series_ids_11_14_2026.json.gz


In [52]:
tv_response = requests.get(
    tv_export_url,
    stream=True,
    timeout=(20, 180),
    headers={"Accept-Encoding": "identity",
             "User-Agent": "CineMatch/1.0"})


tv_response.raise_for_status()

print(f"Status code: {tv_response.status_code}")
print(f"Content length: {tv_response.headers.get('Content-Length')}")

Status code: 200
Content length: 5160974


In [51]:
with gzip.GzipFile(fileobj=tv_response.raw) as gz:
    for i, line in enumerate(gz):
        record = json.loads(line)
        print(record)


        if i ==4:
            break

{'id': 1, 'original_name': 'プライド', 'popularity': 4.3483}
{'id': 2, 'original_name': 'Clerks', 'popularity': 6.6946}
{'id': 3, 'original_name': 'The Message', 'popularity': 0.6214}
{'id': 4, 'original_name': 'The Amazing Mrs Pritchard', 'popularity': 3.027}
{'id': 5, 'original_name': 'La Job', 'popularity': 1.5998}


{'id': 1, 'original_name': 'プライド', 'popularity': 4.3483}
{'id': 2, 'original_name': 'Clerks', 'popularity': 6.6946}
{'id': 3, 'original_name': 'The Message', 'popularity': 0.6214}
{'id': 4, 'original_name': 'The Amazing Mrs Pritchard', 'popularity': 3.027}
{'id': 5, 'original_name': 'La Job', 'popularity': 1.5998}

In [53]:
tv_total_records = 0
tv_unique_ids = set()
tv_duplicate_ids = 0
tv_missing_ids = 0

In [54]:

with gzip.GzipFile(fileobj=tv_response.raw) as gz:
    for line in gz:
        record = json.loads(line)

        tv_total_records += 1

        tv_id = record.get("id")

        if tv_id is None:
            tv_missing_ids += 1

        elif tv_id in tv_unique_ids:
            tv_duplicate_ids += 1

        else:
            tv_unique_ids.add(tv_id)

tv_response.close()


In [55]:

print(f"Total TV records: {tv_total_records:,}")
print(f"Unique TV IDs: {len(tv_unique_ids):,}")
print(f"Duplicate TV IDs: {tv_duplicate_ids:,}")
print(f"Missing TV IDs: {tv_missing_ids:,}")


Total TV records: 236,734
Unique TV IDs: 236,734
Duplicate TV IDs: 0
Missing TV IDs: 0


Total TV records: 236,734
Unique TV IDs: 236,734
Duplicate TV IDs: 0
Missing TV IDs: 0

# Sample testing